# Liquidity Risk

**Purpose:** Demonstrate the liquidity-risk helpers in `finstack.portfolio` on a compact single-name example.

**Prerequisites:** Familiarity with volume, spread, VaR, and market-impact intuition.

**In this notebook:** We simulate prices and volumes, estimate several liquidity metrics, and then roll them into position-level and portfolio-level outputs.


## Concept

Liquidity is multi-dimensional. The same name can look liquid by spread but less liquid by days-to-liquidate or market impact. This notebook deliberately touches several lenses instead of treating liquidity as one scalar.


In [ ]:
import numpy as np

from finstack.portfolio import (
    almgren_chriss_impact,
    amihud_illiquidity,
    days_to_liquidate,
    kyle_lambda,
    liquidity_tier,
    lvar_bangia,
    roll_effective_spread,
)


def simulate_price_path(
    n_days: int = 100,
    initial_price: float = 100.0,
    annual_vol: float = 0.25,
    avg_daily_volume: float = 2_000_000.0,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)
    daily_vol = annual_vol / np.sqrt(252.0)
    eff_returns = rng.normal(0.0, daily_vol, size=n_days)
    half_spread = 0.0005
    bounce = np.where(rng.random(n_days) < 0.5, half_spread, -half_spread)
    log_prices = np.cumsum(eff_returns) + bounce
    prices = initial_price * np.exp(log_prices)
    returns = np.diff(prices) / prices[:-1]
    volumes = rng.lognormal(mean=np.log(avg_daily_volume), sigma=0.35, size=n_days - 1)
    return prices, returns, volumes


## Spread, liquidation, VaR, and impact

The next cell follows the script in the same order an analyst would usually think about the problem: market-data diagnostics first, then liquidation speed, then VaR add-ons and execution cost, and finally a portfolio-level tiering view.


In [ ]:
prices, returns, volumes = simulate_price_path()
print(f"Starting price     : {prices[0]:,.4f}")
print(f"Ending price       : {prices[-1]:,.4f}")
print(f"Mean daily volume  : {volumes.mean():,.0f}")

roll = roll_effective_spread(returns.tolist())
amihud = amihud_illiquidity(returns.tolist(), volumes.tolist())
print(f"\nRoll spread (bps)  : {roll * 1e4:.2f}")
print(f"Amihud illiq.      : {amihud:.4e}")

position_value = 5_000_000.0
participation = 0.10
adv_notional = volumes.mean() * prices[-1]
dtl = days_to_liquidate(position_value, adv_notional, participation)
tier = liquidity_tier(dtl)
print(f"\nADV notional       : ${adv_notional:,.0f}")
print(f"Days to liquidate  : {dtl:.4f}")
print(f"Liquidity tier     : {tier}")

daily_vol = float(returns.std())
var_99 = -2.326 * daily_vol * position_value
lvar = lvar_bangia(
    var=var_99,
    spread_mean=0.0010,
    spread_vol=0.0003,
    confidence=0.99,
    position_value=position_value,
)
print(f"\nBase VaR           : ${abs(lvar['var']):,.2f}")
print(f"Bangia LVaR        : ${abs(lvar['lvar']):,.2f}")
print(f"LVaR / VaR         : {lvar['lvar_ratio']:.4f}x")

shares = position_value / prices[-1]
gamma = roll / (2.0 * volumes.mean()) if np.isfinite(roll) else 1e-8
eta = daily_vol * np.sqrt(prices[-1] / volumes.mean())
impact = almgren_chriss_impact(
    position_size=shares,
    avg_daily_volume=volumes.mean(),
    volatility=daily_vol,
    execution_horizon_days=5.0,
    permanent_impact_coef=gamma,
    temporary_impact_coef=eta,
)
kyle = kyle_lambda(volumes.tolist(), returns.tolist())
print(f"\nAlmgren-Chriss bps : {impact['expected_cost_bps']:.2f}")
print(f"Kyle lambda        : {kyle:.4e}")

portfolio = [
    ("AAPL_5M", 5_000_000.0, 8e9),
    ("SMALL_CAP_5M", 5_000_000.0, 20_000_000.0),
    ("MICRO_CAP_2M", 2_000_000.0, 500_000.0),
    ("MEGA_CAP_50M", 50_000_000.0, 50e9),
]
print("\nPortfolio tiers")
for name, pv, adv in portfolio:
    days = days_to_liquidate(pv, adv, 0.10)
    print(f"  {name:<15} dtl={days:>8.3f}  tier={liquidity_tier(days)}")


## Takeaways

- `roll_effective_spread()` and `amihud_illiquidity()` capture different aspects of trading frictions.
- `days_to_liquidate()` and `liquidity_tier()` translate raw market data into position-management language.
- `lvar_bangia()` and `almgren_chriss_impact()` are natural next steps once you want to fold liquidity into risk or execution decisions.


In [ ]:
{
    "roll_spread_bps": round(roll * 1e4, 2),
    "days_to_liquidate": round(dtl, 4),
    "liquidity_tier": tier,
    "lvar_ratio": round(lvar['lvar_ratio'], 4),
    "impact_bps": round(impact['expected_cost_bps'], 2),
}
